<a href="https://colab.research.google.com/github/juanepstein99/DI_Bootcamp/blob/main/Week15/Day2/DailyChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍰 Daily Challenge — Power Up Your A/B Testing for Your Online Bakery!

## Sweet Bytes: Sample Size & Power Analysis

In this notebook, we will plan an A/B test for **Sweet Bytes**, an online bakery that wants to test a new checkout process.

The current checkout converts approximately **5%** of users, and the team hopes the new checkout can increase conversion to **7%**.

The goal is to understand how many users we need in each group so that the test is statistically reliable.

---

### What we will do

1. Define the A/B test and its hypotheses.
2. Calculate the required sample size for:
   - Effect size = **0.2**
   - Significance level = **0.05**
   - Statistical power = **0.80**
3. Repeat the calculation for effect sizes:
   - 0.1
   - 0.2
   - 0.3
   - 0.4
4. Visualize how effect size changes the required sample size.
5. Explain the relationship in a simple business-friendly way.
6. Add an optional check using the actual conversion rates of 5% and 7%.

## 1. Import the libraries

We will use:

- **NumPy** for numerical work
- **Pandas** for organizing the results
- **Matplotlib** for visualization
- **statsmodels** for statistical power analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

print("Libraries imported successfully.")

# 2. Define the A/B Test

Sweet Bytes currently has a checkout conversion rate of **5%**.

The bakery wants to test a new checkout process that it believes can improve conversion to **7%**.

We will split users into two independent groups:

- **Group A:** Current checkout process
- **Group B:** New checkout process

### Hypotheses

**H₀ (Null Hypothesis):**  
The new checkout process does not create a meaningful improvement compared with the current checkout.

**H₁ (Alternative Hypothesis):**  
The new checkout process performs differently from the current checkout.

For the sample-size calculation requested in the exercise, we will use:

- **Effect size = 0.2**
- **Alpha = 0.05**
- **Power = 0.80**
- **Two-sided test**

## 3. Key Concepts Before the Calculation

### Effect Size

Effect size represents **how large a difference we expect to detect**.

A larger effect is easier to detect, while a smaller effect requires more observations.

### Significance Level — α

We use:

**α = 0.05**

This means we are willing to accept a 5% risk of rejecting the null hypothesis when it is actually true.

### Statistical Power

We use:

**Power = 0.80**

This means that if a real effect exists at the size we planned for, the experiment has an 80% chance of detecting it.

In other words, power helps us avoid running a test that is too small to detect a meaningful difference.

# 4. Calculate the Required Sample Size

The exercise explicitly asks us to calculate the required sample size using an **effect size of 0.2**.

We use `NormalIndPower()` from `statsmodels`.

Because the experiment has two groups of equal size:

- `ratio = 1`
- `alternative = "two-sided"`

In [ ]:
# Create the power-analysis object
power_analysis = NormalIndPower()

# Parameters requested in the challenge
effect_size = 0.2
alpha = 0.05
power = 0.80

# Calculate the sample size needed PER GROUP
required_n = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,
    alternative="two-sided"
)

print(f"Calculated sample size per group: {required_n:.2f}")

### Why do we round up?

The calculation produces a decimal value, but we cannot have a fraction of a user.

We should therefore always **round up** to the next whole number.

Rounding down could slightly reduce the intended statistical power.

In [ ]:
sample_size_per_group = int(np.ceil(required_n))
total_sample_size = sample_size_per_group * 2

print(f"Required sample size per group: {sample_size_per_group:,} users")
print(f"Total users required for the A/B test: {total_sample_size:,} users")

### Result

With:

- Effect size = **0.2**
- Alpha = **0.05**
- Power = **0.80**

Sweet Bytes needs approximately:

- **393 users in Group A**
- **393 users in Group B**

for a total of:

### **786 users**

This is the minimum rounded-up sample size under the assumptions requested in the exercise.

# 5. Analyze the Impact of Effect Size

Now we will calculate the required sample size for four different effect sizes:

- **0.1**
- **0.2**
- **0.3**
- **0.4**

We will keep the other assumptions constant:

- α = 0.05
- Power = 0.80
- Equal group sizes

In [ ]:
effect_sizes = [0.1, 0.2, 0.3, 0.4]

results = []

for es in effect_sizes:
    n = power_analysis.solve_power(
        effect_size=es,
        alpha=0.05,
        power=0.80,
        ratio=1,
        alternative="two-sided"
    )

    # Always round UP to ensure sufficient power
    n_rounded = int(np.ceil(n))

    results.append({
        "Effect Size": es,
        "Sample Size per Group": n_rounded,
        "Total Sample Size": n_rounded * 2
    })

sample_size_df = pd.DataFrame(results)

sample_size_df

### Expected results

The required sample sizes are approximately:

| Effect Size | Sample Size per Group | Total Sample Size |
|---:|---:|---:|
| 0.1 | 1,570 | 3,140 |
| 0.2 | 393 | 786 |
| 0.3 | 175 | 350 |
| 0.4 | 99 | 198 |

The pattern is very clear:

### As effect size increases, the required sample size decreases.

# 6. Visualize Effect Size vs. Sample Size

A graph makes the relationship easier to understand.

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    sample_size_df["Effect Size"],
    sample_size_df["Sample Size per Group"],
    marker="o"
)

plt.title("Effect Size vs. Required Sample Size per Group")
plt.xlabel("Effect Size")
plt.ylabel("Required Sample Size per Group")
plt.grid(alpha=0.3)
plt.show()

## Interpretation

There is an **inverse relationship** between effect size and required sample size.

A **small effect size**, such as 0.1, is difficult to distinguish from normal random variation. Therefore, we need a large amount of data before we can confidently conclude that the difference is real.

A **large effect size**, such as 0.4, creates a more obvious separation between Group A and Group B. Because the signal is stronger, fewer observations are necessary.

### Simple idea

If two cupcakes taste almost identical, you may need many people to decide whether one recipe is truly better.

If one cupcake tastes dramatically better, even a smaller group of tasters will notice the difference.

# 7. Fun Explanation for the Sweet Bytes Team 🍪

Imagine Sweet Bytes wants to decide whether a new chocolate-chip recipe is better.

### Small effect = tiny difference

Suppose the new recipe contains only **one extra chocolate chip**.

The difference is so subtle that some customers will notice it and others will not.

To prove that the recipe is actually better, Sweet Bytes would need to ask **a lot of customers**.

That is similar to an A/B test with a **small effect size**.

---

### Large effect = obvious difference

Now imagine the new cookie has **twice as much chocolate**.

The difference is much easier to notice.

Sweet Bytes does not need thousands of customers to see that people respond differently.

That is similar to an A/B test with a **large effect size**.

---

### Why does the balance matter?

If the sample is **too small**, Sweet Bytes could miss a real improvement and incorrectly conclude that the new checkout does not work.

If the sample is **much larger than necessary**, the bakery may waste:

- advertising budget,
- time,
- traffic,
- development resources,
- and potential sales.

The goal of power analysis is to find the **sweet spot**:

> Enough users to make a confident decision, but not so many that the experiment wastes time and resources.

# 8. Optional Bonus — Compare the Exercise Effect Size With the Actual 5% → 7% Change

The challenge gives us two different pieces of information:

- Current conversion = **5%**
- Expected new conversion = **7%**
- Requested effect size for the main calculation = **0.2**

For the required exercise, we correctly used **0.2**.

However, for a two-proportion conversion-rate test, we can also calculate **Cohen's h** directly from 5% and 7%.

This is useful because the real statistical effect implied by 5% → 7% may not equal 0.2.

In [ ]:
current_conversion = 0.05
new_conversion = 0.07

actual_effect_size = abs(
    proportion_effectsize(current_conversion, new_conversion)
)

print(f"Cohen's h for 5% vs 7%: {actual_effect_size:.4f}")

The effect implied by 5% versus 7% is much smaller than 0.2.

Therefore, if Sweet Bytes truly wants enough power to detect a change specifically from **5% to 7%**, the required sample could be substantially larger than the 393-per-group calculation requested above.

In [ ]:
actual_required_n = power_analysis.solve_power(
    effect_size=actual_effect_size,
    alpha=0.05,
    power=0.80,
    ratio=1,
    alternative="two-sided"
)

actual_required_n_rounded = int(np.ceil(actual_required_n))

print(
    f"Required sample per group for the actual 5% → 7% difference: "
    f"{actual_required_n_rounded:,}"
)

print(
    f"Total sample size: "
    f"{actual_required_n_rounded * 2:,}"
)

### Important note

This bonus does **not replace the answer requested by the assignment**.

The assignment specifically tells us to use an effect size of **0.2**, so the official answer for that part remains approximately **393 users per group**.

The calculation above simply demonstrates why defining effect size carefully is extremely important in real A/B-test planning.

# 9. Final Summary

### Required sample size

Using:

- Effect size = **0.2**
- Alpha = **0.05**
- Power = **0.80**

we need approximately:

### **393 users per group**

or **786 users total**.

---

### Effect-size comparison

As effect size increases:

- 0.1 → about **1,570 users per group**
- 0.2 → about **393 users per group**
- 0.3 → about **175 users per group**
- 0.4 → about **99 users per group**

Therefore:

### **Larger effects require smaller samples, while smaller effects require larger samples.**

This happens because subtle differences are harder to separate from random variation.

---

### Business takeaway

Power analysis helps Sweet Bytes avoid two major problems:

1. **Too little data** → the bakery may make an unreliable decision.
2. **Too much data** → the bakery wastes traffic, money and time.

Choosing an appropriate sample size allows the team to make confident business decisions while using its resources efficiently.

In A/B testing, the goal is not simply to collect as much data as possible.

The goal is to collect **enough data to reliably detect the improvement that actually matters to the business**.

# 10. Reflection

The most important idea I learned from this exercise is that sample size is not an arbitrary number.

It depends directly on:

- the expected effect size,
- the chosen significance level,
- and the desired statistical power.

The biggest challenge is choosing a realistic effect size. A very optimistic effect size can make the required experiment look much smaller than it really needs to be.

For that reason, in a real business experiment I would define the **minimum detectable effect** based on what improvement would actually be valuable to the company and then run the power analysis before launching the test.

This makes the experiment more efficient and makes the final decision more trustworthy.